# 🍽️ ChefMind RAG System
**Hybrid Retrieval-Augmented Generation for Recipe Q&A**

| Step | Component |
|------|----------|
| 1 | Install packages |
| 2 | Load recipe datasets |
| 3 | Chunk + index (FAISS + BM25) |
| 4 | Retrieval functions (hybrid RRF + cross-encoder rerank) |
| 5 | Load ChefMind LoRA model |
| 6 | RAG pipeline |
| 7 | Test |
| 8 | Save indexes to Drive |
| 9 | Expose via ngrok API |

## 📦 Step 1: Install Packages

In [ ]:
%%capture
!pip install datasets langchain langchain-community langchain-core \
             langchain-text-splitters langchain-huggingface \
             faiss-cpu rank-bm25 \
             sentence-transformers pyngrok nest-asyncio fastapi uvicorn \
             transformers peft accelerate bitsandbytes -q
print("✅ Packages installed!")

## 📂 Step 2: Load Recipe Datasets

In [ ]:
from datasets import load_dataset
import pandas as pd

ds1 = load_dataset("tengomucho/simple_recipes", split="train")
ds2 = load_dataset("rahul7star/food-recipes", split="train")
ds3 = load_dataset("formido/recipes", split="train")

def to_text(row):
    return f"Recipe: {row.get('title','')}\nIngredients: {row.get('ingredients','')}\nInstructions: {row.get('directions', row.get('instructions',''))}"

docs1 = [to_text(r) for r in ds1]
docs2 = [to_text(r) for r in ds2]
docs3 = [to_text(r) for r in ds3]
documents = docs1 + docs2 + docs3
print(f"✅ Total: {len(documents)} recipes")

## 🔍 Step 3: Indexing — BM25 + FAISS
*Run once, then save to Drive to skip next session.*

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""],
)
chunks = text_splitter.create_documents(documents)
print(f"✅ {len(chunks)} chunks")

In [ ]:
# ── FAISS dense index ─────────────────────────────────────────────────────
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
print("✅ FAISS ready!")

In [ ]:
# ── BM25 sparse index ────────────────────────────────────────────────────
from rank_bm25 import BM25Okapi

corpus = [c.page_content.lower().split() for c in chunks]
bm25 = BM25Okapi(corpus)
print(f"✅ BM25: {len(corpus)} docs indexed")

## 🔀 Step 4: Retrieval Functions (Hybrid RRF + Cross-Encoder Rerank)

In [ ]:
def generate_queries(q):
    return [q, f"recipe for {q}", f"how to cook {q}", f"ingredients for {q}"]

def multi_query_retrieve(question, k=5):
    queries = generate_queries(question)
    all_docs, seen = [], set()
    for q in queries:
        for doc in vectorstore.similarity_search(q, k=k):
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                all_docs.append(doc)
    return all_docs

def bm25_retrieve(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    top_idx = scores.argsort()[-k:][::-1]
    return [(chunks[i], scores[i]) for i in top_idx]

def rrf_fusion(faiss_docs, bm25_results, k=60):
    scores = {}
    for rank, doc in enumerate(faiss_docs):
        key = doc.page_content[:100]
        scores[key] = scores.get(key, 0) + 1/(k + rank + 1)
    for rank, (doc, _) in enumerate(bm25_results):
        key = doc.page_content[:100]
        scores[key] = scores.get(key, 0) + 1/(k + rank + 1)
    all_docs = {d.page_content[:100]: d for d in faiss_docs}
    all_docs.update({d.page_content[:100]: d for d, _ in bm25_results})
    return [all_docs[k] for k, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True) if k in all_docs]

def hybrid_retrieve(question, k=5):
    return rrf_fusion(vectorstore.similarity_search(question, k=k), bm25_retrieve(question, k=k))

print("✅ Retrieval functions ready!")

In [ ]:
# ── Cross-Encoder reranker ────────────────────────────────────────────────
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_k=5):
    if not docs:
        return docs
    pairs = [[query, doc.page_content] for doc in docs]
    scores = cross_encoder.predict(pairs)
    return [doc for _, doc in sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)[:top_k]]

print("✅ Cross-Encoder ready!")

## 🤖 Step 5: Load ChefMind LoRA Model
Loads the fine-tuned LoRA adapter from HuggingFace on top of the base model.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
LORA_REPO  = "Osamaelbably87/chefmind-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LORA_REPO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"⏳ Loading base model {BASE_MODEL} in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"⏳ Applying LoRA adapter from {LORA_REPO}...")
model = PeftModel.from_pretrained(base_model, LORA_REPO)
model.eval()
print("✅ Model loaded!")

## 🧠 Step 6: RAG Pipeline

In [ ]:
import json, re
from langchain_core.prompts import PromptTemplate

PROMPT_TEMPLATE = """
You are ChefMind, a professional chef AI. Based on the context below, generate a structured recipe.

Context:
{context}

User request: {question}

Respond ONLY with valid JSON in this exact format (no extra text before or after):
{{
    "recipe_name": "...",
    "ingredients": [
        {{"name": "...", "quantity": "..."}}
    ],
    "instructions": [
        "Step 1: ...",
        "Step 2: ..."
    ],
    "cooking_time": "... minutes",
    "difficulty": "Easy/Medium/Hard"
}}
"""

def run_chefmind_pipeline(query: str) -> dict:
    """Full RAG pipeline: retrieve → rerank → generate"""
    # 1. Hybrid retrieve (FAISS + BM25 + RRF)
    retrieved_docs = hybrid_retrieve(query, k=10)

    # 2. Rerank with Cross-Encoder
    reranked_docs = rerank(query, retrieved_docs, top_k=5)

    # 3. Build context from top docs
    context = "\n\n---\n\n".join([doc.page_content for doc in reranked_docs])

    # 4. Build prompt
    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template=PROMPT_TEMPLATE
    ).format(context=context, question=query)

    # 5. Tokenize & generate
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # 6. Parse JSON — try direct then extract first block
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except Exception:
                pass
    return {"error": "Failed to parse JSON", "raw_output": raw}

print("✅ ChefMind Pipeline ready!")

## 🧪 Step 7: Test the Pipeline

In [ ]:
def format_result(result):
    """Pretty-print a pipeline result dict."""
    if "error" in result:
        print("⚠️  Raw output:")
        print(result.get("raw_output"))
        return

    ingredients = result.get('ingredients', '')
    if isinstance(ingredients, list):
        ingredients = ", ".join([
            f"{i.get('name','')} ({i.get('quantity','')})" if isinstance(i, dict) else str(i)
            for i in ingredients
        ])

    instructions = result.get('instructions', '')
    if isinstance(instructions, list):
        instructions = " ".join([f"{idx+1}. {step}" for idx, step in enumerate(instructions)])

    print(f"📛 Recipe:       {result.get('recipe_name')}")
    print(f"🥘 Ingredients:  {ingredients}")
    print(f"👨\u200d🍳 Instructions: {instructions}")
    print(f"⏱️  Time:         {result.get('cooking_time')}")
    print(f"📊 Difficulty:   {result.get('difficulty')}")

In [ ]:
result = run_chefmind_pipeline("pasta with garlic and butter")

print("\n" + "="*50)
format_result(result)
print("="*50)
print("✅ ChefMind Pipeline — COMPLETE!")

## 💾 Step 8: Save Indexes to Google Drive
*Run once. Next session: load from Drive and skip Steps 2–3.*

In [ ]:
import pickle, os

save_dir = "/content/ChefMind"
os.makedirs(save_dir, exist_ok=True)

# Save FAISS
vectorstore.save_local(f"{save_dir}/faiss_index")
print("✅ FAISS saved!")

# Save BM25 + chunks
with open(f"{save_dir}/bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)
with open(f"{save_dir}/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)
print("✅ BM25 + Chunks saved!")

# Verify
for name in os.listdir(save_dir):
    print(f"  📁 {name}")
print(f"\n✅ Saved to: {save_dir}")

In [ ]:
# 💾 Copy to Google Drive (skip rebuilding next session)
from google.colab import drive
import shutil, os

drive.mount("/content/drive")

drive_dir = "/content/drive/MyDrive/ChefMind/indexes"
os.makedirs(drive_dir, exist_ok=True)

shutil.copytree("/content/ChefMind", drive_dir, dirs_exist_ok=True)

# Verify
print("✅ Indexes saved to Drive!")
for f in os.listdir(drive_dir):
    size = os.path.getsize(f"{drive_dir}/{f}") if os.path.isfile(f"{drive_dir}/{f}") else "dir"
    print(f"  📁 {f}: {size}")
print(f"\n📍 Location: {drive_dir}")
print("💡 Next session: load from Drive — skip cells 2–3!")

### ♻️ Step 8b: Load Indexes from Drive (next session)
Run **only** if you already ran Step 8 in a previous session.

In [ ]:
# ── Load from Drive instead of rebuilding ────────────────────────────────
import pickle
from google.colab import drive
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi

drive.mount("/content/drive")

drive_dir = "/content/drive/MyDrive/ChefMind/indexes"

# Load FAISS
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(f"{drive_dir}/faiss_index", embeddings, allow_dangerous_deserialization=True)
print("✅ FAISS loaded!")

# Load BM25 + chunks
with open(f"{drive_dir}/bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)
with open(f"{drive_dir}/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)
print(f"✅ BM25 + {len(chunks)} chunks loaded!")

## 🌐 Step 9: Expose via ngrok API
Publishes ChefMind as a REST endpoint accessible from anywhere.

In [ ]:
from pyngrok import ngrok

# ── Set your ngrok auth token (get free one at https://ngrok.com) ─────────
NGROK_TOKEN = "YOUR_NGROK_AUTH_TOKEN"   # ← replace with your token
# ─────────────────────────────────────────────────────────────────────────

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ NGROK authenticated!")

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import nest_asyncio, uvicorn, threading
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title="ChefMind API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class RecipeRequest(BaseModel):
    query: str

@app.get("/")
def root():
    return {"message": "ChefMind API is running! 🍳"}

@app.post("/recipe")
def get_recipe(request: RecipeRequest):
    result = run_chefmind_pipeline(request.query)

    ingredients = result.get('ingredients', '')
    if isinstance(ingredients, list):
        ingredients = ", ".join([
            f"{i.get('name','')} ({i.get('quantity','')})" if isinstance(i, dict) else str(i)
            for i in ingredients
        ])

    instructions = result.get('instructions', '')
    if isinstance(instructions, list):
        instructions = " ".join([f"{idx+1}. {step}" for idx, step in enumerate(instructions)])

    return {
        "recipe_name":  result.get("recipe_name", ""),
        "ingredients":  ingredients,
        "instructions": instructions,
        "cooking_time": result.get("cooking_time", ""),
        "difficulty":   result.get("difficulty", ""),
    }

# Open ngrok tunnel
ngrok_tunnel = ngrok.connect(8000)
print(f"🌐 API live at: {ngrok_tunnel.public_url}")
print(f"📖 Docs: {ngrok_tunnel.public_url}/docs")

# Start uvicorn in background thread
thread = threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000))
thread.daemon = True
thread.start()
print("✅ Server running!")